In [15]:
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl
from PIL import Image

In [ ]:
def fuzzySystem_OP(pixel, vm):
    
    # Antecedent/Consequent objects
    pixelIN = ctrl.Antecedent(np.arange(0, 2, 1), 'pixelIN')
    grey = ctrl.Antecedent(np.arange(0, 2, 1), 'grey') 
    pixelOUT = ctrl.Consequent(np.arange(0, 2, 1), 'pixelOUT')

    # pixelIN variables
    x_pixelIN = np.arange(0, 2, 1)
    x_grey = np.arange(0, 2, 1)
    x_pixelOUT  = np.arange(0, 2, 1)

    # Generate fuzzy membership functions
    pixelIN_sm = fuzz.trapmf(x_pixelIN, [0, 0, 0.15*vm, 0.6*vm])
    pixelIN_md = fuzz.trimf(x_pixelIN, [0.5*vm, vm, 1.4*vm])
    pixelIN_la = fuzz.trapmf(x_pixelIN, [1.3*vm, 1.6*vm, 1, 1])
    grey_sm = fuzz.trapmf(x_grey, [0, 0, 0.15*vm, 0.6*vm])
    grey_md = fuzz.trimf(x_grey, [0.5*vm, vm, 1.4*vm])
    grey_la = fuzz.trapmf(x_grey, [1.3*vm, 1.6*vm, 1, 1])
    pixelOUT_sm = fuzz.trapmf(x_pixelOUT, [0, 0, 0.15*vm, 0.6*vm])
    pixelOUT_md = fuzz.trimf(x_pixelOUT, [0.5*vm, vm, 1.4*vm])
    pixelOUT_la = fuzz.trapmf(x_pixelOUT, [1.3*vm, 1.6*vm, 1, 1])

    # Activation of fuzzy membership functions at vm using fuzz.interp_membership 
    pixelIN_level_sm = fuzz.interp_membership(x_pixelIN, pixelIN_sm, vm)
    pixelIN_level_md = fuzz.interp_membership(x_pixelIN, pixelIN_md, vm)
    pixelIN_level_la = fuzz.interp_membership(x_pixelIN, pixelIN_la, vm)

    grey_level_sm = fuzz.interp_membership(x_grey, grey_sm, vm)
    grey_level_md = fuzz.interp_membership(x_grey, grey_md, vm)
    grey_level_la = fuzz.interp_membership(x_grey, grey_la, vm)

"""    
    if pixelIN < vm:
        pixelOUT = 
    elif pixelIN == vm:
        pixelOUT = 
    else:
        pixelOUT = 
                        """
    
    # Applying rules 
    # Rule 1 concerns pixel < vm
    # The OR operator means the maximum of these two
    active_rule1 = np.fmax(pixelIN_level_sm, grey_level_sm)

    # Now we apply this by clipping the top off the corresponding pixelOUTput
    # membership function with `np.fmin`
    pixelOUT_activation_sm = np.fmin(active_rule1, pixelOUT_sm)  # removed entirely to 0

    # For rule 2 we connect acceptable service to medium tipping
    pixelOUT_activation_md = np.fmin(grey_level_md, pixelOUT_md)

    # For rule 3 we connect high service OR high food with high tipping
    active_rule3 = np.fmax(pixelIN_level_la, grey_level_la)
    pixelOUT_activation_la = np.fmin(active_rule3, pixelOUT_la)
    pixelOUT0 = np.zeros_like(x_pixelOUT)
    
    # Rules for OP
    rule1 = ctrl.Rule(pixelIN['poor'] or service['poor'], tip['edge'])
    rule2 = ctrl.Rule(service['average'], tip['liver'])
    rule3 = ctrl.Rule(service['good'] or quality['good'], tip['vessel'])
    
    # Control system OP
    tipping_ctrl = ctrl.ControlSystem([rule1, rule2, rule3])
    tipping = ctrl.ControlSystemSimulation(tipping_ctrl)
    
    # Rules for IP
    #rule1 = ctrl.Rule(quality['poor'] or service['poor'], tip['edge'])
    #rule2 = ctrl.Rule(service['average'], tip['liver'])
    #rule3 = ctrl.Rule(service['good'] or quality['good'], tip['vessel'])
    
    # Control system IP
    #tipping_ctrl = ctrl.ControlSystem([rule1, rule2, rule3])
    #tipping = ctrl.ControlSystemSimulation(tipping_ctrl)

    # Inputs
    tipping.input['quality'] = 6.5
    tipping.input['service'] = 9.8

    # Computation
    tipping.compute()
    
    return(pixelOUT)

In [21]:
print(fuzz.interp_membership.__doc__)


    Find the degree of membership ``u(xx)`` for a given value of ``x = xx``.

    Parameters
    ----------
    x : 1d array
        Independent discrete variable vector.
    xmf : 1d array
        Fuzzy membership function for ``x``.  Same length as ``x``.
    xx : float or array of floats
        Value(s) on universe ``x`` where the interpolated membership is
        desired.
    zero_outside_x : bool, optional
        Defines the behavior if ``xx`` contains value(s) which are outside the
        universe range as defined by ``x``.  If `True` (default), all
        extrapolated values will be zero.  If `False`, the first or last value
        in ``x`` will be what is returned to the left or right of the range,
        respectively.

    Returns
    -------
    xxmf : float or array of floats
        Membership function value at ``xx``, ``u(xx)``.  If ``xx`` is a single
        value, this will be a single value; if it is an array or iterable the
        result will be returned as a 

In [ ]:
    # Normalize image
    imagem = Image.open('image')
    imagem = Image.fromarray(normalize(imagem).astype('uint16')) # VER NORMALIZE, AINDA NAO FUNCIONA
    #imagem.save('normalized.png')